# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

> NOTE: DO NOT RUN THESE CELLS IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY

In [ ]:
#!pip install -qU ragas==0.2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.7/175.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.6/411.6 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.8/454.8 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/1

In [ ]:
#!pip install -qU langchain-community==0.3.14 langchain-openai==0.2.14 unstructured==0.16.12 langgraph==0.2.61 langchain-qdrant==0.2.0

In [1]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [2]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [3]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - and download our webpages which we'll be using for our data today.

These webpages are from [Simon Willison's](https://simonwillison.net/) yearly "AI learnings".

- [2023 Blog](https://simonwillison.net/2023/Dec/31/ai-in-2023/)
- [2024 Blog](https://simonwillison.net/2024/Dec/31/llms-in-2024/)

Let's start by collecting our data into a useful pile!

In [4]:
!mkdir data

mkdir: cannot create directory ‘data’: File exists


In [5]:
!curl https://simonwillison.net/2023/Dec/31/ai-in-2023/ -o data/2023_llms.html

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 31427    0 31427    0     0  57446      0 --:--:-- --:--:-- --:--:-- 57453


In [6]:
!curl https://simonwillison.net/2024/Dec/31/llms-in-2024/ -o data/2024_llms.html

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 70286    0 70286    0     0  1238k      0 --:--:-- --:--:-- --:--:-- 1247k


Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [7]:
from langchain_community.document_loaders import DirectoryLoader

path = "data/"
loader = DirectoryLoader(path, glob="*.html")
docs = loader.load()

[nltk_data] Downloading package punkt to /home/tabesink/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/tabesink/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [8]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

/home/tabesink/Projects/code/AIE5/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
/home/tabesink/Projects/code/AIE5/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/lang/arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
/home/tabesink/Projects/code/AIE5/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/lang/persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [9]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [10]:
from ragas.testset.graph import Node, NodeType

for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 2, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [11]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/2 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/12 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/26 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 14, relationships: 67)

We can save and load our knowledge graphs as follows.

In [12]:
kg.save("ai_across_years_kg.json")
ai_across_years_kg = KnowledgeGraph.load("ai_across_years_kg.json")
ai_across_years_kg

KnowledgeGraph(nodes: 14, relationships: 67)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [13]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=ai_across_years_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [14]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

The three query synthesizers generate different types of test questions:

1. SingleHopSpecificQuerySynthesizer (50% of queries):
    - Generates straightforward, single-fact questions
    - Only requires looking up one piece of information
    - Example: "What year was GPT-4 released?"

2. MultiHopAbstractQuerySynthesizer (25% of queries): 
    - Creates questions requiring connecting multiple facts
    - Focuses on high-level concepts and relationships
    - Example: "How has the development of LLMs impacted AI ethics?"

3. MultiHopSpecificQuerySynthesizer (25% of queries):
    - Also requires connecting multiple pieces of information
    - But asks about specific details rather than abstract concepts
    - Example: "What features of Claude 3 improved over Claude 2?"


Finally, we can use our `TestSetGenerator` to generate our testset!

In [15]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What advancements in LLMs were made in 2024?,[Code may be the best application The ethics o...,The sequel to the blog post 'Things we learned...,single_hop_specifc_query_synthesizer
1,How do LLMs handle JavaScript code generation?,[Based Development As a computer scientist and...,LLMs are particularly effective at writing cod...,single_hop_specifc_query_synthesizer
2,Wht are the key advancemnts and chllenges of G...,[Simon Willison’s Weblog Subscribe Stuff we fi...,"In 2023, Large Language Models (LLMs) like GPT...",single_hop_specifc_query_synthesizer
3,What OpenAI do in 2023?,[easy to follow. The rest of the document incl...,"In 2023, OpenAI was frequently discussed in th...",single_hop_specifc_query_synthesizer
4,What is the significance of Apple's MLX librar...,[Prompt driven app generation is a commodity a...,"Apple's MLX library is noted as excellent, des...",single_hop_specifc_query_synthesizer
5,How do the challenges of LLMs as black boxes i...,[<1-hop>\n\nCode may be the best application T...,The challenges of LLMs as black boxes signific...,multi_hop_abstract_query_synthesizer
6,How do the challenges of understanding and con...,[<1-hop>\n\nCode may be the best application T...,The challenges of understanding and controllin...,multi_hop_abstract_query_synthesizer
7,What challenges have been identified with GPT-...,[<1-hop>\n\nCode may be the best application T...,Despite significant advancements in Large Lang...,multi_hop_abstract_query_synthesizer
8,How has Google's Gemini series contributed to ...,[<1-hop>\n\nday after that. DeepSeek v3 is a h...,Google's Gemini series has significantly contr...,multi_hop_specific_query_synthesizer
9,What are some key advancements in Large Langua...,[<1-hop>\n\nneeds guidance. Those of us who un...,"In 2024, Simon Willison highlighted several ke...",multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [16]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/2 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/12 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/26 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [17]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,Wut is Anthropic's role in the development of ...,[Code may be the best application The ethics o...,Anthropic is one of the organizations that hav...,single_hop_specifc_query_synthesizer
1,What are the current challenges and ethical co...,[Based Development As a computer scientist and...,The current challenges and ethical concerns as...,single_hop_specifc_query_synthesizer
2,What insights about Large Language Models were...,[Simon Willison’s Weblog Subscribe Stuff we fi...,Simon Willison's Weblog highlights that 2023 w...,single_hop_specifc_query_synthesizer
3,Wht are the implicashuns of LLMs on human soci...,[easy to follow. The rest of the document incl...,The impact of LLMs on human society is already...,single_hop_specifc_query_synthesizer
4,How have advancements in Large Language Models...,[<1-hop>\n\nCode may be the best application T...,Advancements in Large Language Models (LLMs) h...,multi_hop_abstract_query_synthesizer
5,What are the environmental impacts and trainin...,[<1-hop>\n\nCode may be the best application T...,The environmental impact of Large Language Mod...,multi_hop_abstract_query_synthesizer
6,How have advancements in Large Language Models...,[<1-hop>\n\nCode may be the best application T...,Advancements in Large Language Models (LLMs) h...,multi_hop_abstract_query_synthesizer
7,How do the advancements in Large Language Mode...,[<1-hop>\n\nCode may be the best application T...,Advancements in Large Language Models (LLMs) h...,multi_hop_abstract_query_synthesizer
8,How has the development and deployment of Larg...,[<1-hop>\n\nCode may be the best application T...,The development and deployment of Large Langua...,multi_hop_specific_query_synthesizer
9,How did the developments in Large Language Mod...,[<1-hop>\n\neasy to follow. The rest of the do...,"In 2023, Large Language Models (LLMs) experien...",multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [18]:
from langsmith import Client

client = Client()

dataset_name = "State of AI Across the Years!"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="State of AI Across the Years!"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [19]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [20]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [21]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [22]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [23]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="State of AI"
)

In [24]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [25]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

For our LLM, we will be using TogetherAI's endpoints as well!

We're going to be using Meta Llama 3.1 70B Instruct Turbo - a powerful model which should get us powerful results!

In [26]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

Finally, we can set-up our RAG LCEL chain!

In [27]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [28]:
rag_chain.invoke({"question" : "What are Agents?"})

'Agents are described as "AI systems that can go away and act on your behalf." However, the term is considered vague, lacking a single, clear, and widely understood definition. There are different interpretations, such as those viewing agents as things that perform tasks on behalf of users, and others thinking in terms of language models (LLMs) given tools to solve problems. There is also skepticism about their utility due to challenges related to gullibility in LLMs, which can lead to difficulties in making meaningful decisions.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4o as our evaluation LLM for our base Evaluators.

In [29]:
eval_llm = ChatOpenAI(model="gpt-4o")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [30]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

dope_or_nope_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "dopeness": "Is this submission dope, lit, or cool?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`: Evaluates the quality of question answering by checking if the model's response correctly answers the given question based on the provided context.

- `labeled_helpfulness_evaluator`: Evaluates how helpful the response is to the user by comparing the model's output against a reference answer, taking into account the original question.

- `dope_or_nope_evaluator`: Evaluates whether the response has a "cool" or engaging style by checking if the submission is "dope, lit, or cool".

## LangSmith Evaluation

In [31]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dope_or_nope_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'left-stem-80' at:
https://smith.langchain.com/o/63374c27-c836-4974-b50b-437920f0fd97/datasets/bd69909a-1a58-45e7-8d72-ad755846cb3e/compare?selectedSessions=0bcd8c7a-d172-4ac1-85b2-8a72da667944




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,How have advancements in GPT-4 and other large...,I don't know.,None,"In 2024, advancements in GPT-4 and other large...",0,0,0,1.272117,4bec1e42-588e-4e2f-939a-13e89c319a6f,5fbef30a-7a58-4fea-8f04-ab097aa0cadc
1,What advancements in Large Language Models wer...,The advancements observed in Large Language Mo...,None,"Between 2023 and 2024, significant advancement...",1,0,1,3.037942,81f04f6d-719b-47b3-8842-92f385063e88,969f422f-5fb5-42f2-ac95-8be9d0f52060
2,How did the developments in Large Language Mod...,The developments in Large Language Models (LLM...,None,"In 2023, Large Language Models (LLMs) experien...",0,0,1,7.255147,8e9e5d7c-da99-4ede-ba33-229b566ae92f,18670860-ce58-4c56-9933-5861f337a00b
3,How has the development and deployment of Larg...,The development and deployment of Large Langua...,None,The development and deployment of Large Langua...,1,1,0,2.445558,b68f7fa5-f9e2-4633-91f1-0ac45d141f7e,9afc6ff7-78f4-4484-abe0-e6e066efd04a
4,How do the advancements in Large Language Mode...,I don't know.,None,Advancements in Large Language Models (LLMs) h...,0,0,0,0.914013,37b845bc-2b56-437d-9924-2a76c0ddeaec,51cee9b9-aa88-4dad-b55b-9e90b9651c08
5,How have advancements in Large Language Models...,Advancements in Large Language Models (LLMs) h...,None,Advancements in Large Language Models (LLMs) h...,1,1,0,5.551966,533b9e29-d675-4248-8dfd-c49dd0b2cd94,0a8bc48c-23a6-4deb-8ac9-6e9ed6210e6c
6,What are the environmental impacts and trainin...,The environmental impact of training Large Lan...,None,The environmental impact of Large Language Mod...,0,0,0,2.006569,3423b4c6-357d-4b47-8487-0323063e011e,86982317-ce38-460b-93aa-02b89cd728a8
7,How have advancements in Large Language Models...,The advancements in Large Language Models (LLM...,None,Advancements in Large Language Models (LLMs) h...,1,0,0,2.307908,b7743f42-1548-47a3-99c0-de3992ab6830,fd3a132f-b8ac-4d7e-ab95-b21976fcdbd1
8,Wht are the implicashuns of LLMs on human soci...,I don't know.,None,The impact of LLMs on human society is already...,0,0,0,1.144254,c5b405fb-fd55-4e85-93f1-314ec31a9082,d60f2338-4568-4594-90b5-cf7987bf0c39
9,What insights about Large Language Models were...,In Simon Willison's Weblog regarding the devel...,None,Simon Willison's Weblog highlights that 2023 w...,1,1,1,2.874709,6d5c78ef-c39b-47b4-b025-106f6111a8fe,906158c8-101c-413a-917e-428aef6fd466


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [32]:
DOPE_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

You must answer the questions in a dope way, be cool!

Context: {context}
Question: {question}
"""

dope_rag_prompt = ChatPromptTemplate.from_template(DOPE_RAG_PROMPT)

In [33]:
rag_documents = docs

In [34]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

Modifying chunk size affects performance in several key ways:

1. Larger chunks provide more context in each segment, which can help the model better understand the full meaning and relationships within the text
2. However, larger chunks also mean fewer total chunks, which could reduce the granularity of retrieval matches
3. Smaller chunks allow for more precise matching to specific parts of documents, but may miss broader context
4. The chunk overlap helps maintain continuity between segments, but needs to be balanced with chunk size


In [35]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

Modifying the embedding model affects performance in several ways:

1. Different embedding models have varying capabilities in capturing semantic meaning
2. More advanced models (like text-embedding-3-large) generally produce higher quality embeddings that better represent the text
3. The dimensionality and quality of embeddings directly impacts the accuracy of vector similarity searches
4. Better embeddings lead to more relevant document retrieval during the RAG process
5. However, more sophisticated embedding models may have higher computational costs and latency

In [36]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="AI Across Years (Augmented)"
)

In [37]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [38]:
dope_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dope_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [39]:
dope_rag_chain.invoke({"question" : "what are Agents?"})

'Agents are kinda like digital sidekicks—AI systems that are expected to act on your behalf, but here\'s the twist: the term "agents" is super vague. There are two main camps: some folks see them as travel agent types doing stuff for you, while others think of LLMs using tools to solve problems. Yet, despite all the chatter, real examples running in the wild are pretty rare, mostly because AI systems struggle with gullibility. So, until we nail that down, agent dreams might just stay dreams for a while! 🕶️✨'

Finally, we can evaluate the new chain on the same test set!

In [40]:
evaluate(
    dope_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dope_or_nope_evaluator
    ],
    metadata={"revision_id": "dope_chain"},
)

View the evaluation results for experiment: 'fixed-bill-6' at:
https://smith.langchain.com/o/63374c27-c836-4974-b50b-437920f0fd97/datasets/bd69909a-1a58-45e7-8d72-ad755846cb3e/compare?selectedSessions=82f70831-eba9-4beb-9547-e2ffc6e911fe




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,How have advancements in GPT-4 and other large...,"Yo, in 2024, the advancements in GPT-4 and oth...",None,"In 2024, advancements in GPT-4 and other large...",0,0,1,3.850270,4bec1e42-588e-4e2f-939a-13e89c319a6f,1df1992b-afb5-442c-b1b2-c46717730631
1,What advancements in Large Language Models wer...,"Yo, let me break it down for you! Between 2023...",None,"Between 2023 and 2024, significant advancement...",1,1,1,5.253585,81f04f6d-719b-47b3-8842-92f385063e88,732e0d50-83c3-464d-9262-74b27675849f
2,How did the developments in Large Language Mod...,"Yo, the development of Large Language Models (...",None,"In 2023, Large Language Models (LLMs) experien...",1,0,1,5.848975,8e9e5d7c-da99-4ede-ba33-229b566ae92f,5c175c08-8241-4882-b166-14093fc9de28
3,How has the development and deployment of Larg...,"Yo, it's a wild ride with LLMs! So, companies ...",None,The development and deployment of Large Langua...,1,1,1,3.087652,b68f7fa5-f9e2-4633-91f1-0ac45d141f7e,39449fc9-5400-41ba-9d20-3de8640f1eb2
4,How do the advancements in Large Language Mode...,"Yo, the advancements in LLMs are a double-edge...",None,Advancements in Large Language Models (LLMs) h...,1,0,1,4.421644,37b845bc-2b56-437d-9924-2a76c0ddeaec,437a9e64-4f68-4124-afc8-992b3cf589ef
5,How have advancements in Large Language Models...,"Yo, check it out! Advancements in Large Langua...",None,Advancements in Large Language Models (LLMs) h...,1,0,1,3.127004,533b9e29-d675-4248-8dfd-c49dd0b2cd94,bb513e8d-3c1d-4b3e-ba9f-b6053f299152
6,What are the environmental impacts and trainin...,"Yo, check it! The environmental impact of Larg...",None,The environmental impact of Large Language Mod...,1,0,1,4.337192,3423b4c6-357d-4b47-8487-0323063e011e,da832c8d-a0dd-4716-8425-44fe5318084d
7,How have advancements in Large Language Models...,Advancements in Large Language Models (LLMs) h...,None,Advancements in Large Language Models (LLMs) h...,0,0,1,4.844697,b7743f42-1548-47a3-99c0-de3992ab6830,6aaa1035-6262-40fd-b777-c9e88973d0eb
8,Wht are the implicashuns of LLMs on human soci...,"Yo, the implications of LLMs on human society ...",None,The impact of LLMs on human society is already...,1,0,1,2.945297,c5b405fb-fd55-4e85-93f1-314ec31a9082,564bbc02-32ae-453e-8a0f-7c6b9205ec48
9,What insights about Large Language Models were...,"Yo, here’s the lowdown from Simon Willison’s W...",None,Simon Willison's Weblog highlights that 2023 w...,1,0,1,4.340505,6d5c78ef-c39b-47b4-b025-106f6111a8fe,cfa40a40-0193-489c-a8be-aa8ab960f071


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

## Comparing Chain Performance

After making several changes to our RAG chain, we observed some interesting changes in the evaluation metrics:

1. Answer Relevance likely improved due to:
    - The `text-embedding-3-large` model providing more accurate semantic matching
    - Larger chunks giving more complete context to the LLM

2. Helpfulness scores increased because:
    - The "dope" prompt encourages more engaging, relatable responses
    - Larger context chunks allow for more comprehensive answers

3. "Dopeness" metrics showed improvement through:
    - Direct prompt engineering for casual, accessible language
    - Better context retrieval enabling more natural explanations

The combination of better embeddings, more context, and targeted prompting created a more effective and engaging RAG system.

![Comparison of Chain Performance](Capture-1.JPG)

![Comparison of Chain Performance](Capture-2.JPG)